In [5]:
"""
!pip install numpy pandas scikit-learn matplotlib tqdm
!pip install git+https://github.com/moment-timeseries-foundation-model/moment.git
""";

## Import script and packages

In [ ]:
from pathlib import Path
import sys

THIS_DIR = Path.cwd().resolve()
sys.path.insert(0, str(THIS_DIR))

In [11]:
import moment_utils

model = moment_utils.import_moment_model_classifier()
model.init()

MOMENT takes 3 inputs:

- An input time series of length timesteps and channels, and

- Two optional masks, both of length

    - The input mask is utilized to regulate the time steps or patches that the model should attend to. For instance, in the case of shorter time series, you may opt not to attend to padding. To implement this, you can provide an input mask with zeros in the padded locations.
        
    - The second mask, referred to simply as mask, denotes masked or unobserved values. We employ mask tokens to replace all patches containing any masked time step (for further details, refer to Section 3.2 in our paper). MOMENT can attend to these mask tokens during reconstruction.

By default, all time steps are observed and attended to.


In [13]:
from pprint import pprint
import torch

# takes in tensor of shape [batchsize, n_channels, context_length]
x = torch.randn(16, 38, 512)
output = model(x_enc=x)
pprint(output)

TimeseriesOutputs(forecast=None,
                  anomaly_scores=None,
                  logits=tensor([[-0.0661, -0.1754],
        [-0.0531, -0.1838],
        [-0.0722, -0.1154],
        [-0.0864, -0.1606],
        [-0.0359, -0.1707],
        [-0.0527, -0.1545],
        [-0.0340, -0.1592],
        [-0.0493, -0.1711],
        [-0.0372, -0.1791],
        [-0.0918, -0.1665],
        [-0.0729, -0.1673],
        [-0.0594, -0.1512],
        [-0.0352, -0.1558],
        [-0.0441, -0.1811],
        [-0.0143, -0.1161],
        [-0.0677, -0.2184]], grad_fn=<AddmmBackward0>),
                  labels=None,
                  input_mask=None,
                  pretrain_mask=None,
                  reconstruction=None,
                  embeddings=tensor([[[-0.0558, -0.0792, -0.0183,  ..., -0.0703, -0.0512, -0.0498],
         [-0.0489, -0.0363,  0.0185,  ...,  0.0828, -0.0684,  0.0171],
         [-0.0198, -0.0846, -0.0636,  ..., -0.2514, -0.1694,  0.0315],
         ...,
         [-0.0642, -0.1157, 

In [ ]:
# Define a data loader 
# create a fake dataset for demonstration
from torch.utils.data import DataLoader, TensorDataset
from torch import nn
from tqdm import tqdm

dataset = TensorDataset(torch.randn(10, 38, 250), torch.randint(0, 2, (10,)))

train_dataloader = DataLoader(dataset, batch_size=1, shuffle=True) 
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# use tqdm for progress bar
train_dataloader = tqdm(train_dataloader, desc="Training")
for data, labels in train_dataloader:
    # forward [batch_size, n_channels, forecast_horizon]
    output = model(x_enc=data)

    # backward
    loss = criterion(output.logits, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    train_dataloader.set_postfix(loss=loss.item())

Training: 100%|██████████| 10/10 [00:15<00:00,  1.60s/it, loss=0.673]
